# Module 01 — Models

In Module 00 you made your first call with `.invoke()`. Now you'll learn the **other ways to talk to a model** and *when* to use each:

- **`invoke`** — one complete answer.
- **`stream`** — the answer arrives piece by piece (the "typing" effect).
- **`batch`** — many prompts at once, run in parallel.

You'll also **switch providers by changing one string**, and **tune behavior** with parameters like `temperature`.

Run each code cell with **Shift + Enter**, top to bottom.

## 0. Setup — load your key

Same as Module 00: load the `.env` from the project root and create one model we'll reuse. Nothing new here — run it and move on.

In [1]:
import os
from dotenv import load_dotenv

# .env lives in the project root (one folder up from this module).
load_dotenv(dotenv_path=os.path.join("..", ".env"))
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-2.5-flash")

print("Key loaded:", bool(os.getenv("GOOGLE_API_KEY")))
print("Model ready:", model.__class__.__name__)

Key loaded: True
Model ready: ChatGoogleGenerativeAI


## 1. `invoke` — one complete answer

`invoke` sends your prompt and **waits for the whole reply**, then hands it back as one `AIMessage`.

Use it when you need the full answer before you can do anything with it — scoring, parsing, saving to a database. The trade-off: the user sees **nothing** until the model is completely done.

In [2]:
response = model.invoke("Name three planets in our solar system.")

print(response.text)

Here are three planets in our solar system:

1.  **Earth**
2.  **Mars**
3.  **Jupiter**


## 2. `stream` — answer piece by piece

`stream` gives you the answer in small chunks **as the model produces it**, instead of waiting for the end. Each chunk is a fragment of text; you print them as they arrive.

This is exactly the "typing" effect you see in ChatGPT. Same total time, but the user starts reading immediately — so the app *feels* far faster. Reach for `stream` whenever a human is watching the output live.

In [3]:
# end="" so the chunks join into flowing text instead of one-per-line.
for chunk in model.stream("Explain what a token is, in two short sentences."):
    print(chunk.text, end="", flush=True)

A token is a fundamental unit of text, representing a word, part of a word, or punctuation. Large language models break down input text into these tokens for processing and generating responses.

## 3. `batch` — many prompts at once

Got a *list* of prompts? Don't loop and `invoke` one at a time — that runs them back-to-back. `batch` takes a list and runs them **in parallel**, returning a list of answers in the same order.

Fewer lines than a `for` loop, and much faster when you have many independent prompts (labeling rows, summarizing documents, generating variations).

In [4]:
prompts = [
    "Give me a fun fact about the Moon.",
    "Give me a fun fact about the ocean.",
    "Give me a fun fact about honey.",
]

answers = model.batch(prompts)

for prompt, answer in zip(prompts, answers):
    print("Q:", prompt)
    print("A:", answer.text)
    print("-" * 40)

Q: Give me a fun fact about the Moon.
A: Here's a fun one:

Because the Moon has no atmosphere, there's no wind, rain, or erosion to wear things away. This means the footprints left by the Apollo astronauts on the Moon are still there, perfectly preserved, and will likely remain visible for millions of years!
----------------------------------------
Q: Give me a fun fact about the ocean.
A: Here's a fun one:

The ocean is so vast and mysterious that we've actually explored less than **5%** of it! That means over 95% of Earth's largest living space remains undiscovered, potentially hiding millions of unknown species and incredible geological features.
----------------------------------------
Q: Give me a fun fact about honey.
A: Here's a fun fact about honey:

**Honey is the only food that never spoils!** Archaeologists have found pots of honey in ancient Egyptian tombs that are over 3,000 years old and still perfectly edible. Its low water content, high acidity, and the presence of hyd

## 4. Switch providers — change one string

This is the framework paying off. `init_chat_model` takes a `"provider:model"` string. Want a different provider? **Change the string, not your code.** `invoke` / `stream` / `batch` all keep working.

```python
init_chat_model("google_genai:gemini-2.5-flash")   # Google
init_chat_model("openai:gpt-4o-mini")               # OpenAI  (needs OPENAI_API_KEY)
init_chat_model("groq:llama-3.3-70b-versatile")     # Groq    (needs GROQ_API_KEY)
```

Each provider needs its own API key in `.env`. The cell below stays on Gemini so it runs for everyone — but notice the *only* thing that would change is the string.

In [5]:
# Same three verbs, any provider. Swap the string when you have another key.
other_model = init_chat_model("google_genai:gemini-2.5-flash")

print(other_model.invoke("Say hello in French.").text)

Bonjour!


## 5. Model parameters — steer the behavior

You can pass settings into `init_chat_model` to control *how* the model responds. The two you'll use most:

- **`temperature`** — randomness/creativity. `0` = focused and repeatable (good for facts, code, extraction). Higher (≈`0.9`) = more varied and creative (good for brainstorming, writing).
- **`max_tokens`** — a hard cap on how long the answer can be (controls cost and length).

Run this to feel the difference: the same creative prompt at low vs. high temperature.

In [7]:
focused = init_chat_model("google_genai:gemini-2.5-flash", temperature=0)
creative = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.9)

prompt = "Invent a name for a coffee shop on Mars."

# print("temperature=0   ->", focused.invoke(prompt).text)
print("temperature=0.9 ->", creative.invoke(prompt).text)

temperature=0.9 -> Here are some names for a coffee shop on Mars, playing on different aspects of the planet, space, and the future:

**Evocative & Martian-Themed:**

1.  **Red Sands Cafe:** Simple, direct, and conjures the Martian landscape.
2.  **Olympus Perk:** A play on Olympus Mons, the massive Martian volcano.
3.  **Gale Crater Coffee:** Named after a prominent Martian crater.
4.  **Phobos & Deimos Brews:** After Mars's two moons.
5.  **Ares Aroma:** "Ares" is the Greek god equivalent to Roman Mars.
6.  **Martian Roast:** Classic and clear.
7.  **The Solstice Sip:** "Solstice" implies a moment in the planet's orbit, "Sol" is a Martian day.
8.  **Valles Marineris Brews:** After the grand canyon system on Mars.

**Futuristic & Space-Themed:**

9.  **The Zenith Cafe:** Zenith refers to the point directly above an observer.
10. **Orbit & Grind:** Implies planetary motion and coffee preparation.
11. **Cosmic Cup:** Simple, catchy, and celestial.
12. **Stardust Cafe:** Evokes a sense o

## Recap

- **`invoke`** — one full answer; use when you need the complete result before acting.
- **`stream`** — chunks as they're generated; use when a human watches output live.
- **`batch`** — a list of prompts run in parallel; use for many independent prompts.
- **Switch providers** by changing the `"provider:model"` string — your code stays the same.
- **Parameters** like `temperature` (randomness) and `max_tokens` (length) steer the model.

## 🛠️ Try it yourself

1. Take any prompt and call it three ways — `invoke`, `stream`, `batch` (batch with a list of one) — and confirm you get the same kind of answer.
2. Stream a **longer** request (e.g. "Explain gravity to a 10-year-old in a full paragraph"). Watch the words appear — *that's* why streaming matters.
3. In `batch`, add two more prompts. Notice you didn't write a loop or wait three separate times.
4. Set `temperature=0` and run the Mars coffee-shop prompt **twice**. Then set `temperature=0.9` and run it twice. Which one repeats itself? Why?
5. **Stretch:** add `max_tokens=20` to a model and watch a long answer get cut short.

When you're done, say **"next"** and we'll build **Module 02 — Messages & Prompts**.